# 04 — Winner-derived TTF baseline

This notebook builds a compact, two-stage ExtraTrees baseline on the existing FabJudge window features. It does not rerun raw-data preprocessing, windowing, normalization, or feature selection.

**Scope:** This is a baseline reproduction layer, not a claim that the complete PHM 2018 winning system has been reproduced. The repository's 03 notebook selected one shared set of 20 candidates from the three saved target rankings; the same saved set is used for each target because no target-specific feature lists were saved. The official challenge page documents the three TTF failure modes and null labels outside the specified horizon, but it does not provide a near-failure threshold. Therefore this notebook defines near-failure as the lowest 25% of non-null training TTF values for each target, measured within the fixed training groups. The threshold is learned from training labels only.

For inference, Stage 1 classifies near-failure windows. Stage 2 predicts TTF for windows classified as near-failure. A window classified as normal receives the threshold boundary as its combined prediction. The models and this combination rule are simple benchmark choices, not verified reproductions of proprietary winner settings.

The official challenge description is available at [PHM Society — PHM Data Challenge 2018](https://phmsociety.org/conference/annual-conference-of-the-phm-society/annual-conference-of-the-prognostics-and-health-management-society-2018-b/phm-data-challenge-6/).

## 1. Load existing artifacts and define the baseline

Read the split artifact emitted by 03 and the saved compact candidate list. Train sampling follows 03's deterministic group-balanced quotas, capped at 100,000 usable training windows per target. Validation remains the complete fixed grouped validation set.

In [ ]:
from pathlib import Path
import gc
import json
import math
import time
import warnings

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
from sklearn.ensemble import ExtraTreesClassifier, ExtraTreesRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    mean_absolute_error,
    mean_squared_error,
    precision_score,
    r2_score,
    recall_score,
)
from sklearn.pipeline import make_pipeline


def find_project_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "data" / "interim" / "phm2018_jev" / "features").is_dir():
            return candidate
    raise FileNotFoundError("Could not find existing PHM2018 engineered features.")


PROJECT_ROOT = find_project_root()
ARTIFACT_DIR = PROJECT_ROOT / "artifacts" / "random_forest"
FEATURE_TRAIN_DIR = PROJECT_ROOT / "data" / "interim" / "phm2018_jev" / "features" / "train"
SPLIT_PATH = ARTIFACT_DIR / "ttf_baseline_group_split.csv"
SPLIT_METADATA_PATH = ARTIFACT_DIR / "ttf_baseline_split_metadata.json"
SELECTED_FEATURE_PATH = ARTIFACT_DIR / "jev_candidate_features.csv"
MODEL_DIR = PROJECT_ROOT / "artifacts" / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42
MAX_TRAIN_ROWS_PER_TARGET = 100_000
CSV_CHUNK_ROWS = 50_000
NEAR_FAILURE_QUANTILE = 0.25
N_ESTIMATORS = 200
MIN_SAMPLES_LEAF = 5
CLASSIFIER_MAX_FEATURES = "sqrt"
REGRESSOR_MAX_FEATURES = 1.0

print("Project:", PROJECT_ROOT)
print("scikit-learn:", sklearn.__version__)
print("Feature files:", FEATURE_TRAIN_DIR)
print("Model output:", MODEL_DIR)

## 2. Load the fixed split and selected feature list

The 03 notebook's saved artifact uses `source_file::sequence_index` as the group key. The compact feature list is the actual 20-row `jev_candidate_features.csv`; this notebook does not rank or select features again.

In [ ]:
if not SPLIT_PATH.is_file() or not SPLIT_METADATA_PATH.is_file():
    raise FileNotFoundError(
        "The fixed split artifact is missing. Run the existing split cell in "
        "notebooks/03_rf_feature_check.ipynb once to persist its assignments."
    )
if not SELECTED_FEATURE_PATH.is_file():
    raise FileNotFoundError(f"Missing saved 03 compact feature list: {SELECTED_FEATURE_PATH}")

split_metadata = json.loads(SPLIT_METADATA_PATH.read_text(encoding="utf-8"))
group_table = pd.read_csv(SPLIT_PATH, low_memory=False)
selected_features = pd.read_csv(SELECTED_FEATURE_PATH)["feature"].astype(str).tolist()
TARGET_COLUMNS = list(split_metadata["target_columns"])
TARGET_LABELS = dict(split_metadata["target_labels"])

expected_group_columns = {
    "group_id", "source_file", "Tool", "total_windows", "split", *TARGET_COLUMNS
}
missing_group_columns = expected_group_columns.difference(group_table.columns)
if missing_group_columns:
    raise ValueError(f"Split artifact is missing columns: {sorted(missing_group_columns)}")
required_selected_features = set(selected_features)
if len(selected_features) != 20 or len(required_selected_features) != len(selected_features):
    raise ValueError(f"Expected the saved 20 unique compact candidates; found {len(selected_features)}")

train_groups = set(group_table.loc[group_table["split"].eq("train"), "group_id"].astype(str))
validation_groups = set(group_table.loc[group_table["split"].eq("validation"), "group_id"].astype(str))
group_table["group_id"] = group_table["group_id"].astype(str)
if train_groups & validation_groups:
    raise AssertionError("Train/validation group leakage detected in the saved 03 split.")
if len(train_groups) != int(split_metadata["train_groups"]):
    raise AssertionError("Saved train group count does not match 03 metadata.")
if len(validation_groups) != int(split_metadata["validation_groups"]):
    raise AssertionError("Saved validation group count does not match 03 metadata.")
if int(split_metadata["random_seed"]) != SEED:
    raise ValueError("The saved split seed differs from this baseline's expected seed 42.")

split_by_group = dict(zip(group_table["group_id"], group_table["split"]))
train_group_table = group_table.loc[group_table["split"].eq("train")]
validation_group_table = group_table.loc[group_table["split"].eq("validation")]
print("Targets:", TARGET_COLUMNS)
print("Split seed:", split_metadata["random_seed"])
print("Compact feature count:", len(selected_features))
for target in TARGET_COLUMNS:
    print(f"\n{target}: {len(selected_features)} shared compact features")
    print(selected_features)

## 3. Verify leakage-free grouping

Print the counts from the persisted split and assert that no group appears on both sides. Window counts below cover all rows in each group; target-usable windows are reported after loading labels.

In [ ]:
assert train_groups.isdisjoint(validation_groups)
assert train_groups | validation_groups == set(group_table["group_id"])
assert int(split_metadata["group_overlap"]) == 0

print("number of train groups:", len(train_groups))
print("number of validation groups:", len(validation_groups))
print("number of train windows:", int(train_group_table["total_windows"].sum()))
print("number of validation windows:", int(validation_group_table["total_windows"].sum()))
print("group overlap =", len(train_groups & validation_groups))
display(
    group_table.groupby("split")
    .agg(groups=("group_id", "nunique"), windows=("total_windows", "sum"))
    .reindex(["train", "validation"])
)

## 4. Load compact training samples and full validation data

Train rows use the same cap and per-group `linspace` selection policy as 03. Validation is read in full. Only the 20 selected features, target labels, and the identifiers needed for diagnostics are loaded from each processed CSV.

In [ ]:
def allocate_quotas(group_counts, cap):
    counts = {str(group_id): int(count) for group_id, count in group_counts.items() if int(count) > 0}
    budget = min(int(cap), sum(counts.values()))
    quotas = {group_id: 0 for group_id in counts}
    active = list(counts)
    remaining = budget
    while active and remaining > 0:
        per_group, extra = divmod(remaining, len(active))
        if per_group == 0:
            for group_id in active[:remaining]:
                quotas[group_id] += 1
            break
        saturated = [
            group_id for group_id in active
            if counts[group_id] - quotas[group_id] <= per_group
        ]
        if saturated:
            for group_id in saturated:
                available = counts[group_id] - quotas[group_id]
                quotas[group_id] += available
                remaining -= available
            saturated_set = set(saturated)
            active = [group_id for group_id in active if group_id not in saturated_set]
        else:
            for position, group_id in enumerate(active):
                quotas[group_id] += per_group + int(position < extra)
            remaining = 0
    return {group_id: quota for group_id, quota in quotas.items() if quota > 0}


sample_plan = {}
sample_cursors = {}
for target in TARGET_COLUMNS:
    counts = (
        train_group_table[["group_id", target]]
        .set_index("group_id")[target]
        .astype(int)
    )
    quotas = allocate_quotas(counts.to_dict(), MAX_TRAIN_ROWS_PER_TARGET)
    sample_plan[target] = {
        group_id: np.linspace(0, int(counts[group_id]) - 1, num=quota, dtype=np.int64)
        for group_id, quota in quotas.items()
    }
    sample_cursors[target] = {}

train_parts = {target: [] for target in TARGET_COLUMNS}
validation_parts = {target: [] for target in TARGET_COLUMNS}
train_full_counts = {target: 0 for target in TARGET_COLUMNS}
validation_full_counts = {target: 0 for target in TARGET_COLUMNS}

train_paths = sorted(FEATURE_TRAIN_DIR.glob("*.csv"))
if not train_paths:
    raise FileNotFoundError(f"No processed train feature CSVs found in {FEATURE_TRAIN_DIR}")
first_header = pd.read_csv(train_paths[0], nrows=0).columns.tolist()
for path in train_paths[1:]:
    if pd.read_csv(path, nrows=0).columns.tolist() != first_header:
        raise ValueError(f"Processed CSV header differs from the first file: {path.name}")

required_columns = [
    "source_file", "sequence_index", "window_index", *selected_features, *TARGET_COLUMNS
]
missing_columns = set(required_columns).difference(first_header)
if missing_columns:
    raise ValueError(f"Processed feature CSVs are missing columns: {sorted(missing_columns)}")
train_meta_columns = ["group_id", "source_file", "sequence_index", "window_index"]
val_meta_columns = ["group_id", "source_file", "sequence_index", "window_index"]
for file_number, path in enumerate(train_paths, start=1):
    print(f"Reading existing features {file_number}/{len(train_paths)}: {path.name}")
    for chunk in pd.read_csv(
        path, usecols=required_columns, chunksize=CSV_CHUNK_ROWS, low_memory=False
    ):
        chunk["group_id"] = (
            chunk["source_file"].astype(str) + "::" + chunk["sequence_index"].astype(str)
        )
        chunk["split"] = chunk["group_id"].map(split_by_group)
        if chunk["split"].isna().any():
            unseen = chunk.loc[chunk["split"].isna(), "group_id"].head(3).tolist()
            raise ValueError(f"Feature rows reference groups absent from 03 split: {unseen}")
        feature_values = chunk[selected_features].replace([np.inf, -np.inf], np.nan).astype(np.float32)
        chunk[selected_features] = feature_values

        for target in TARGET_COLUMNS:
            finite_target = np.isfinite(pd.to_numeric(chunk[target], errors="coerce"))
            train_mask = chunk["split"].eq("train") & finite_target
            train_candidates = chunk.loc[train_mask, [*train_meta_columns, *selected_features, target]]
            train_full_counts[target] += len(train_candidates)
            for group_id, group_rows in train_candidates.groupby("group_id", sort=False):
                cursor = sample_cursors[target].get(group_id, 0)
                ordinals = cursor + np.arange(len(group_rows), dtype=np.int64)
                wanted = sample_plan[target].get(group_id)
                if wanted is not None:
                    keep = np.isin(ordinals, wanted, assume_unique=True)
                    if keep.any():
                        train_parts[target].append(group_rows.iloc[np.flatnonzero(keep)].copy())
                sample_cursors[target][group_id] = cursor + len(group_rows)

            validation_mask = chunk["split"].eq("validation") & finite_target
            validation_rows = chunk.loc[
                validation_mask, [*val_meta_columns, *selected_features, target]
            ].copy()
            validation_full_counts[target] += len(validation_rows)
            if not validation_rows.empty:
                validation_parts[target].append(validation_rows)

train_data = {}
validation_data = {}
sampling_rows = []
for target in TARGET_COLUMNS:
    train_data[target] = pd.concat(train_parts[target], ignore_index=True)
    validation_data[target] = pd.concat(validation_parts[target], ignore_index=True)
    expected_train = sum(len(indices) for indices in sample_plan[target].values())
    if len(train_data[target]) != expected_train:
        raise AssertionError(
            f"Sample count mismatch for {target}: expected {expected_train}, "
            f"found {len(train_data[target])}"
        )
    expected_validation = int(validation_group_table[target].sum())
    if validation_full_counts[target] != expected_validation:
        raise AssertionError(
            f"Validation label count mismatch for {target}: artifact has "
            f"{expected_validation}, CSV scan has {validation_full_counts[target]}"
        )
    if train_full_counts[target] != int(train_group_table[target].sum()):
        raise AssertionError(f"Training label count mismatch for {target}")
    sampling_rows.append({
        "target": TARGET_LABELS.get(target, target),
        "total usable train windows": train_full_counts[target],
        "actual model train windows": len(train_data[target]),
        "full validation windows": len(validation_data[target]),
        "train groups represented": train_data[target]["group_id"].nunique(),
        "validation groups represented": validation_data[target]["group_id"].nunique(),
        "sampling method": "03 group-balanced quotas; equidistant row ordinals within group",
        "random seed": SEED,
    })
    train_data[target][target] = pd.to_numeric(train_data[target][target], errors="raise").astype(np.float64)
    validation_data[target][target] = pd.to_numeric(
        validation_data[target][target], errors="raise"
    ).astype(np.float64)

sampling_summary = pd.DataFrame(sampling_rows)
display(sampling_summary)

## 5. Define the two-stage ExtraTrees baseline

For each target, Stage 1 labels windows at or below the training-sample 25th percentile as near-failure. Stage 2 is trained only on those near-failure training windows. Classifier imbalance is handled with balanced class weights. The two forest configurations are fixed and modest; there is no hyperparameter search.

In [ ]:
CLASSIFIER_PARAMETERS = {
    "n_estimators": N_ESTIMATORS,
    "random_state": SEED,
    "n_jobs": -1,
    "class_weight": "balanced",
    "min_samples_leaf": MIN_SAMPLES_LEAF,
    "max_features": CLASSIFIER_MAX_FEATURES,
}
REGRESSOR_PARAMETERS = {
    "n_estimators": N_ESTIMATORS,
    "random_state": SEED,
    "n_jobs": -1,
    "min_samples_leaf": MIN_SAMPLES_LEAF,
    "max_features": REGRESSOR_MAX_FEATURES,
}

print("Near-failure definition: TTF <= training-only 25th percentile")
print("Classifier parameters:", CLASSIFIER_PARAMETERS)
print("Regressor parameters:", REGRESSOR_PARAMETERS)
print("Feature missing/non-finite values are replaced by train-fitted median imputers.")


def fit_target(target):
    train_frame = train_data[target]
    validation_frame = validation_data[target]
    feature_names = selected_features
    y_train = train_frame[target].to_numpy(dtype=np.float64)
    y_validation = validation_frame[target].to_numpy(dtype=np.float64)
    threshold = float(np.quantile(y_train, NEAR_FAILURE_QUANTILE))
    near_train = y_train <= threshold
    near_validation = y_validation <= threshold
    if near_train.sum() == 0 or (~near_train).sum() == 0:
        raise ValueError(f"Near-failure threshold did not create two classes for {target}")

    X_train = train_frame[feature_names].to_numpy(dtype=np.float32, copy=True)
    X_validation = validation_frame[feature_names].to_numpy(dtype=np.float32, copy=True)
    if np.isinf(X_train).any() or np.isinf(X_validation).any():
        raise ValueError(f"Infinite feature values remain after preprocessing for {target}")

    classifier = make_pipeline(
        SimpleImputer(strategy="median"),
        ExtraTreesClassifier(**CLASSIFIER_PARAMETERS),
    )
    regressor = make_pipeline(
        SimpleImputer(strategy="median"),
        ExtraTreesRegressor(**REGRESSOR_PARAMETERS),
    )

    started = time.perf_counter()
    classifier.fit(X_train, near_train.astype(np.int8))
    classifier_seconds = time.perf_counter() - started
    started = time.perf_counter()
    regressor.fit(X_train[near_train], y_train[near_train])
    regressor_seconds = time.perf_counter() - started

    predicted_near = classifier.predict(X_validation).astype(bool)
    combined_prediction = np.full(len(validation_frame), threshold, dtype=np.float64)
    if predicted_near.any():
        near_prediction = regressor.predict(X_validation[predicted_near])
        combined_prediction[predicted_near] = np.clip(near_prediction, 0.0, threshold)

    cm = confusion_matrix(near_validation.astype(np.int8), predicted_near.astype(np.int8), labels=[0, 1])
    rmse = float(np.sqrt(mean_squared_error(y_validation, combined_prediction)))
    metrics = {
        "target": TARGET_LABELS.get(target, target),
        "target_column": target,
        "n_features": len(feature_names),
        "n_train_usable": int(train_full_counts[target]),
        "n_model_train": len(train_frame),
        "n_stage2_train": int(near_train.sum()),
        "n_validation": len(validation_frame),
        "near_failure_threshold": threshold,
        "near_failure_quantile": NEAR_FAILURE_QUANTILE,
        "train_near_failure_fraction": float(near_train.mean()),
        "validation_near_failure_fraction": float(near_validation.mean()),
        "classifier_precision": float(precision_score(near_validation, predicted_near, zero_division=0)),
        "classifier_recall": float(recall_score(near_validation, predicted_near, zero_division=0)),
        "classifier_f1": float(f1_score(near_validation, predicted_near, zero_division=0)),
        "classifier_accuracy_reference": float(accuracy_score(near_validation, predicted_near)),
        "confusion_matrix": cm.tolist(),
        "MAE": float(mean_absolute_error(y_validation, combined_prediction)),
        "RMSE": rmse,
        "R2": float(r2_score(y_validation, combined_prediction)),
        "classifier_fit_seconds": float(classifier_seconds),
        "regressor_fit_seconds": float(regressor_seconds),
    }

    slug = target.removeprefix("ttf_").removesuffix("_seconds")
    classifier_path = MODEL_DIR / f"ttf_{slug}_classifier.joblib"
    regressor_path = MODEL_DIR / f"ttf_{slug}_regressor.joblib"
    joblib.dump(classifier, classifier_path, compress=3)
    joblib.dump(regressor, regressor_path, compress=3)

    prediction_frame = validation_frame[
        ["group_id", "source_file", "sequence_index", "window_index"]
    ].copy()
    prediction_frame["actual_ttf"] = y_validation
    prediction_frame["predicted_ttf"] = combined_prediction
    prediction_frame["absolute_error"] = np.abs(y_validation - combined_prediction)
    prediction_frame["actual_near_failure"] = near_validation
    prediction_frame["predicted_near_failure"] = predicted_near

    print(f"\n{TARGET_LABELS.get(target, target)}")
    print(f"Features: {len(feature_names)}; Stage-2 train windows: {int(near_train.sum()):,}")
    print(f"Training-only near-failure threshold: {threshold:,.3f} seconds")
    print(
        "Stage-1 precision={:.4f}, recall={:.4f}, F1={:.4f}, accuracy(reference)={:.4f}".format(
            metrics["classifier_precision"], metrics["classifier_recall"],
            metrics["classifier_f1"], metrics["classifier_accuracy_reference"]
        )
    )
    print("Confusion matrix (rows=actual [normal, near], columns=predicted [normal, near]):")
    display(pd.DataFrame(cm, index=["actual normal", "actual near"], columns=["pred normal", "pred near"]))
    print(
        "Combined validation MAE={:,.3f}, RMSE={:,.3f}, R²={:.4f}".format(
            metrics["MAE"], metrics["RMSE"], metrics["R2"]
        )
    )
    print(f"Models saved: {classifier_path.name}; {regressor_path.name}")

    result = {
        "metrics": metrics,
        "selected_features": list(feature_names),
        "classifier_parameters": dict(CLASSIFIER_PARAMETERS),
        "regressor_parameters": dict(REGRESSOR_PARAMETERS),
        "classifier_path": str(classifier_path.relative_to(PROJECT_ROOT)),
        "regressor_path": str(regressor_path.relative_to(PROJECT_ROOT)),
        "predictions": prediction_frame,
    }
    del classifier, regressor, X_train, X_validation
    gc.collect()
    return result

## 6. Train target 1 — FlowCool pressure low

In [ ]:
RESULTS = {}
RESULTS[TARGET_COLUMNS[0]] = fit_target(TARGET_COLUMNS[0])

## 7. Train target 2 — FlowCool pressure high

In [ ]:
RESULTS[TARGET_COLUMNS[1]] = fit_target(TARGET_COLUMNS[1])

## 8. Train target 3 — FlowCool leak

In [ ]:
RESULTS[TARGET_COLUMNS[2]] = fit_target(TARGET_COLUMNS[2])

## 9. Validation metrics summary

In [ ]:
metrics_table = pd.DataFrame([RESULTS[target]["metrics"] for target in TARGET_COLUMNS])
display(metrics_table[
    ["target", "n_features", "classifier_precision", "classifier_recall", "classifier_f1",
     "classifier_accuracy_reference", "MAE", "RMSE", "R2", "n_model_train", "n_validation"]
].rename(columns={"n_features": "selected features"}))

## 10. Prediction sanity checks and hard validation cases

The sample rows are distributed across the validation file order for a quick sanity check. Error cases are the ten largest absolute errors per target. The plot is a visual check of the combined predictions.

In [ ]:
for target in TARGET_COLUMNS:
    prediction_frame = RESULTS[target]["predictions"]
    positions = np.linspace(
        0, len(prediction_frame) - 1, num=min(12, len(prediction_frame)), dtype=np.int64
    )
    print(f"\n{TARGET_LABELS.get(target, target)} — validation sample")
    display(prediction_frame.iloc[positions][
        ["actual_ttf", "predicted_ttf", "absolute_error", "group_id", "source_file", "sequence_index"]
    ].reset_index(drop=True))
    print(f"{TARGET_LABELS.get(target, target)} — largest absolute errors")
    display(
        prediction_frame.nlargest(10, "absolute_error")[[
            "group_id", "actual_ttf", "predicted_ttf", "absolute_error"
        ]].reset_index(drop=True)
    )

fig, axes = plt.subplots(1, len(TARGET_COLUMNS), figsize=(16, 5), squeeze=False)
for axis, target in zip(axes[0], TARGET_COLUMNS):
    prediction_frame = RESULTS[target]["predictions"]
    positions = np.linspace(
        0, len(prediction_frame) - 1, num=min(5_000, len(prediction_frame)), dtype=np.int64
    )
    plot_frame = prediction_frame.iloc[positions]
    axis.scatter(plot_frame["actual_ttf"], plot_frame["predicted_ttf"], s=8, alpha=0.25)
    lower = min(plot_frame["actual_ttf"].min(), plot_frame["predicted_ttf"].min())
    upper = max(plot_frame["actual_ttf"].max(), plot_frame["predicted_ttf"].max())
    axis.plot([lower, upper], [lower, upper], color="black", linewidth=1)
    axis.set_title(TARGET_LABELS.get(target, target))
    axis.set_xlabel("Actual TTF (seconds)")
    axis.set_ylabel("Predicted TTF (seconds)")
fig.suptitle("Combined actual vs predicted TTF — validation sample")
fig.tight_layout()
plt.show()

## 11. Save metadata and final summary

In [ ]:
metadata = {
    "experiment": "Winner-derived two-stage ExtraTrees TTF baseline",
    "scope": "baseline reproduction layer; not the complete PHM 2018 winning system",
    "random_seed": SEED,
    "split_artifact": str(SPLIT_PATH.relative_to(PROJECT_ROOT)),
    "split_metadata_artifact": str(SPLIT_METADATA_PATH.relative_to(PROJECT_ROOT)),
    "group_definition": split_metadata["group_definition"],
    "train_groups": sorted(train_groups),
    "validation_groups": sorted(validation_groups),
    "train_group_count": len(train_groups),
    "validation_group_count": len(validation_groups),
    "group_overlap": len(train_groups & validation_groups),
    "feature_selection_artifact": str(SELECTED_FEATURE_PATH.relative_to(PROJECT_ROOT)),
    "feature_selection_scope": "shared 20-feature compact candidate set saved by 03; reused for all targets",
    "selected_features": selected_features,
    "train_sampling": {
        "total_usable_train_windows": train_full_counts,
        "actual_model_train_windows": {target: len(train_data[target]) for target in TARGET_COLUMNS},
        "method": "03 group-balanced quotas, capped at 100000 rows per target; np.linspace ordinals within each group",
        "random_seed": SEED,
    },
    "validation": {
        "method": "all finite target rows from the persisted 03 grouped validation groups",
        "rows_by_target": validation_full_counts,
    },
    "near_failure_definition": {
        "method": "target-specific training-sample TTF quantile",
        "quantile": NEAR_FAILURE_QUANTILE,
        "positive_label": "TTF <= threshold",
        "normal_prediction_combination": "use the near-failure threshold as the combined TTF estimate",
    },
    "classifier_parameters": CLASSIFIER_PARAMETERS,
    "regressor_parameters": REGRESSOR_PARAMETERS,
    "targets": {
        target: {
            "display_name": TARGET_LABELS.get(target, target),
            "selected_features": RESULTS[target]["selected_features"],
            "classifier_parameters": RESULTS[target]["classifier_parameters"],
            "regressor_parameters": RESULTS[target]["regressor_parameters"],
            "classifier_model": RESULTS[target]["classifier_path"],
            "regressor_model": RESULTS[target]["regressor_path"],
            "metrics": RESULTS[target]["metrics"],
        }
        for target in TARGET_COLUMNS
    },
    "reproduced": [
        "TTF prediction task on existing feature pipeline",
        "two-stage tree-based baseline",
        "leakage-safe fixed grouped validation",
        "target-wise baseline performance",
    ],
    "not_reproduced": [
        "complete proprietary PHM 2018 winning pipeline",
        "all winner-specific hyperparameters and threshold rules",
        "leaderboard submission environment",
        "FabJudge JEV, retrieval, or LLM Judge",
    ],
}
METADATA_PATH = MODEL_DIR / "ttf_baseline_metadata.json"
METADATA_PATH.write_text(json.dumps(metadata, indent=2, allow_nan=False), encoding="utf-8")
print("Saved model metadata:", METADATA_PATH)
print("Model artifacts:")
for target in TARGET_COLUMNS:
    print(" -", RESULTS[target]["classifier_path"])
    print(" -", RESULTS[target]["regressor_path"])
print("\nSummary metrics:")
display(metrics_table[[
    "target", "n_features", "classifier_f1", "MAE", "RMSE", "R2"
]].rename(columns={"n_features": "selected features"}))

print("\nReproduced: TTF prediction, existing deterministic feature pipeline, two-stage tree baseline, fixed grouped validation, target-wise performance.")
print("Not reproduced: full proprietary winner pipeline, all original hyperparameters, leaderboard environment, JEV/retrieval/LLM Judge.")